# P05: calibrate the cost model on a Colab or Kaggle GPU (Phase 2.3)

- **Colab:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**.
- **Kaggle:** File → Import Notebook (this file); Session options: **Accelerator → GPU T4 x2**
  and **Internet on**; then **Run All** (or Save Version → Save & Run All).

About 10 minutes, most of it installing the NVIDIA HPC SDK.

It measures what the cost model needs (memory bandwidth, arithmetic rate, OpenMP
startup and thread-team cost on the host; kernel launch, host↔device copies, bandwidth,
arithmetic rate and device startup on the GPU) and prints a JSON machine file, named
after the platform (`colab-t4` or `kaggle-t4`). **Copy the JSON the last cell prints back
to Claude**; it becomes `machine/<name>.json`, used as `p05tool -machine=machine/<name>.json`.

Generated by `eval/calibrate/make_notebook.py` from `eval/calibrate/calibrate.{c,py}`:
edit those, not this notebook.


In [ ]:
# Where to work: Colab's /content or Kaggle's /kaggle/working; the machine
# file is named after the platform (colab-t4 / kaggle-t4).
import base64, gzip, json, os
KAGGLE = os.path.isdir('/kaggle/working')
ROOT = '/kaggle/working' if KAGGLE else '/content'
os.makedirs(ROOT, exist_ok=True)
os.environ['P05_ROOT'] = ROOT  # the %%bash cells cd here
os.environ['P05_PLATFORM'] = 'kaggle' if KAGGLE else 'colab'
FILES = json.loads(gzip.decompress(base64.b64decode("""
H4sIAAAAAAAC/71beXPbRrL/KhO6kgAyBVHykUSyvOUrm61nWy7H+8crUQWBwFDEClcwgI7V43ff
X/fM4CBBSs46T4lFYjDd09N394zuRmGQxLMyqKQXjg7FaG9PfIjDMp/JLFykQXmpxDwvRbWQIsxV
JdI8ksmPSqRBuIgzKYqgDFJZyVIJ59MiUFIceE9cb5oBUYu6uBWzOk4iBUSxGouyzpSQQCGadYRD
a6gqKKu6EHkmlYgzMS+lWjCyosxDqZRUrgiySFyXcSUbMvZeZKDipfcvlWdHQkkpPr979fbDOy+N
mBTGIFqCxAte96U4DcoLdSb4pyjjrFK0tMjqdCZLfAVBUV5XDY7fwINDzFVVKYN0X+yZbx8xVpVx
EIkZqLuOo2oxFvvYLV5GmBUkyVj8/fWeMpTon7xQhAIfH7tje0oEShSTZ1WeJ2B7TWSBO+kAyhWM
87iEkPb0px/PJ0QXWG7ZB8npKeckuCSRyblw4rkzcd2xqPu4SnkRgwO9nyATMi2q2xae1OMcnAol
KwkkW8WpFFiyLhqUb+VVjAkOKUElruNqIXbffpo887+8+vz3d1/cw97ClSax9zO8CWjLhazO10iv
kqCGfIdJv5RlBrobkiNNXJ/iBlOYQ3lfzG6hbS9Flf/fvMzTHiM0DSIKquAc+lgUcXYBvAFsJchu
e8iGfhizcMDLPIRmjgUvSEp4HdyOyQDkumgqrXUQdAV9MWN99dN6RSrc7hH64nmeVhnG95oXJ3ui
uUEptbbJiIFAwIrhd5WRETifynweV8EsTuLq1guLwj1kGE1NldfhAiscPBWvRQGLkolMZVYJJ8u1
Ce/ajTO6qgzm8zh0jxgH0WSEBeE8fyrSOqniIrndDSK4EoKvZCZgnzCMg595PutWljO2JM8Ls6FS
BGnOcpGp9jS9/QTi8vFjUJuP4acqEVe8ZyUyeVNhmLFlOXQwuxgLlWO+5ROvcZ3XSQR3NU/It7AZ
4AvEdr2IQ+2+zq05y6sg2Wsc0V7H/Z7D4oq8rNj6Y3wQbug6eQRwCpNgjnBnj+IsTOoITixPC2/x
sjsCdxXn62NYY20Q3u5iZZAsl4cwGMk5+fdfP7xyblxxI47xb0dMvF9++UU8xudksn/Um/YzzTPT
v/ajh+j5U/Pi5//is8V48ul3/9O7z/679+8+kJrQ7hBmqjgUcO2zBN4+v/Zr5VzlceSKu2nG7r0O
IXnwQxUyhL0d0XAIVb30Ye30wnnz/uTN//gfTj6efDn5+I83Y/FDpVyeV8qqLmF4yquufAX4HbEv
n4NteiSjoT0MPcHs5To9O2wTjor/Lf1KZGM7fmWos9MKCCXVczMsQfPzuaPfupqSeC6c7woDh/DA
MW7uQCVkWY7FdATjEflcpDLNy9spaBlpQCHkTVw5+/ppSb8oD7BExVh6coSPFwIBN3782NVAxWl8
hldX2KslxInF9+KnHmMKu28YxhdYizZytepstFkhmh+uBkNr7XApjKQxD9rLOd74GuX5j9qKYDUk
Xe2VnIbN5GzB+4ahs9WBcAxwuI3McJBZwCObGBBoBszog8zkGQQT4sHumMloCWxoYSfWXzzOvmJ1
K2EDTPYaZ3pdGrZmpZ+wmCbzxkq3q4Utp3wE+T/NrUdFGVykgYCTEt1kQSiEhKhOpKPXc78NY7vE
E3+7pN/P3G9G7LeQg7EK5BU1BUtFSk0JQZMrI+4gbiQe7ZodFWWxfd8AaZGAdkLGrHdKrDoyC7yW
ig3/HDFHneuEHI/zMWXdtfJWfdIM8x34DopRSOCZzc7O3HW6FOwg5PWeb/oea4atwutNJkcNGxml
5mIJLhJ2fFtnZDXBJOOorYeag5FH/VmdSWIXQOY1ucEK6GeG8YJJqVrfZhzTbEibwBtfO461vRmH
crP7EvzG7xn/Dvl3hixmuQETa+cGbPzuazBC2wdwdXzMCrLNaDaQZd8M4+lgQvoTQwpGGCrOLo/W
o9sCDt7X+StLXxc0yghcW6nWV9KW9+LFC3Hw7KgTlZkyMqw7E/vGYgLFax72uw8HeGCtb3SkVk1k
NEuLYyxkNKNW5GRI2fcn41ZgCO5G1xCnpNXNvjxphqAkVldoXMpuRUpQLWLWQ+IZ+QMvOM2QIjw5
64bNg6feBF6PXgCjTiBoQV1XLjewmtTjoXyefC2fP/7z/fs/y+BnY6t3m7lrVe+HxtKH4L+WkTY1
6aWGQ4zV1Xjrll/Rirtc4M4DkMrFgM5a0gKqX8IvZz8iypSoCvS4cdHeupmQUP4ty5y937rwdAmv
i3AWIKp5Izx6ujFgVpQ0lpMoCWVXHOuekyRjkQ0GPpqQZ0ZgN48fa7Y2whmC0XNXXgZVnsahdbkW
kZHQskvjoN+2Muzl1MMpM9fillmdRL7lTUZrgChK3/00uPGNblqe4H1Up+nt6cGz5xSS7ybLfpS6
1lHqGmFkf0JfTJDalDqYGGXBu6lC1U1shFmX0mRe+zFShlaR2x1QbASSg4mJnxvle39g/YtpNoLT
cyZ90+uJ2YWxMWVGqI/iOao20XaGVuWsmy0+20ZXzlr7fXJ45M228ae7d9O7SYPC4b7OoUXC+7zr
oCTNiZUfZ3EVB4mv+ymOjoD3qLG2Jo12cyFmSDFNtzLIbOuG4LYXZttNw7BMd8RWedZ6kSE9P9ig
5obWCiFciSimXsKsruQDVWm/r0k3pDyTLQq//40V/i+l/kFq3k29oHt+lTvb0uadgLwgkq+jgchN
idiwVlNHklW7yqHYwenkMDvTer3cQAfZwF9KiTGyLbT0lZZ6obrS4jbpmNWiyvtdmtU8RXdU93SJ
5va7It2shX9Ps4092WEMOir0jUrnfaaPXBeDWqxxvjwWjsm1nrvib+IZuHEw6WdPNr/RlRaE9zej
JJhrpdRmO51Ep+9p60F1A5GbipkhOVMVqb9yMam/hl8n/XsN7dsW/7TDwZKouz9qezU7jA39s//H
fa2Ut1eDfYKr1T7B1Wqf4C1ZFnZ/u9LpF0FY5kp31ChdjWyPzdtgaaYgW0ud/vJK7F7HdePN2F2g
5vRC7Ti6voQVv/EmhqmNkg/VDVSDNTP+2/prm/ciLbyXoU+/Wcn1YE4+iH9E/Iaqy7z+1hUXs/KR
zKJ4Tiwl35kGsa59sJ1wLMJFUIqdHTzYVjjlVvSOkpUtyVWtggt5uPH8t5deWWGvZiR6cfKGtP7p
/llXAFdttx2iDNPCmdG65pgYyI0lk513GyD7nbJ3A/THzdCTbdB0uDwASnLdvwfu4wa4retxTt4H
7JWw9wPTefUWBFup1nP60L2icBuwKQX70JtqxYEaZSPeAZ70KpitROnkYhC6yTu2gFPuBGDxww9s
PdSDebqOiTMs1OmJw1p9cAYPY/HwyJMzxpYTHduW0yo5SK3R1q3gUK9BWFI75rrxC/0W0YClZ5dI
vbPOvZLvFZv3WMw2W7hBMx197z2fm+lXvSOriXZPGB91r7XQlZlH3+3VqtybxdmezK5EcVst8uwJ
IaH/PshA1aV80A2a5mKLCMSufT+PE+mRQ+Tab9PpcXErdnfpEoxIn+Ib5Tf2bkz6lO/F6NTWaarb
ByEL8ySY7VaE8qKo8TsMRXYViin9NKkzhudJgNA1He2eHID04phn49dxGP70bDpaIcmiZcJob6/1
7aDOabhw1q6JcELFhLjDV4jAv2kGTYivpE595rEktM1lIJEgW1Oe+N1cM9Kn/IERD9KkOOAjxJ/s
xaPm1pFI46zWSBUSv+h2l0K+ESkg+IXmmOn4HYG/srzlSwOstAhpGsHMnL0omhAkeitxpuJI8jmP
WdQTn4NrcxVJiYtc1LQ38nXB9XTkWQUD2pSuDZCVQ5d4HT2A6C8p+WsGNLfNQ66ar0USVOBt2gyo
RSJv2idKbhTymxZC1TNDZTt0236v4HpJb0m0v737/I7cqfKKoFp4UVySZjn2OZgp+nR8nwB8n3zA
r5/f/f4bYH4S4pGWRHv9i6+R2GtiqTYuOiWbsh6RZwY7HRhjUN4it6f4Phawy+OPYC1dNGIPAOTt
FjyCOG1AKDmCCMecjblwf2FQwAdIHxpc1NXxl7KWqMvkjfk6WEKyfWHVu50d7BTf4jJHArez45CP
gB7fLd2l25xOlZ72MyGcg/gOxcJhixWM9bjnM5+O7izbZoGSzEdNtrsUdz+KH71/5UiY+hvAq3kA
1kaCdGWNSCB17rrLL91DgQHtUj0qbgrHXcI595znPMkDlKievh9np7mtFFhs98pBIzvdIDL8c9no
fTqVLIPsQjqsG+5ZZ6HYRLEip/t7FrU2jvdQkGpX/VHDxhX35+iKThbeooy0VX57ZSqHPWpLRsnP
56GBOcFJCI/YFx/i18JRdOmCrwdK7TzwaPG6R2ZTSji10nf+XM9aKl+RqCiXPqW4V+nd0TfaoN4A
qcOMGgX7+kjmTINRyZDIzAG4kUQ6Yy1OnZnFwmyiCdiVCRhpZSZVmpF2qd4kleSFxDyHJgLdLnC7
SM8BhK+rVDLoiib1AXfEwRpJLtlyrc2XON9TAORYTkqLaUp2gGZMd4yoX4Oaa8+Mc53QSp6rAyvv
oNCZObtA75U5O/9ET6UTSRVCQemGyLHvR3no+5vNtmMb8IlBhdDsh0mg1HGDHo75bYvyN5kUv9qp
tmNReEEU+fYMH6mFjqiUWZTyjzouZcTuY8t8unj6FdPDkGaDMUGdVMet1/GQR2HCmzf0ejrCVrKL
NtcaxMTRvIvNBPY5pJClReMMhoARngkyCJnbVMXk8J0Iibz3BXiFwSBRufXenbbFVsRZDnvtE9XO
J+EXHkuHoJA02pTpOi+pLrUhyUsvI/ruFKWcxzdAUkyeIe9oUGk/1IlY7FMJC3PPZijN/BBpBew5
8EJUqDscOT1VJPDXGGJGQot3nFNsoZPLTEdnZOiBR4kSp7SnSL17Otlbn6Job30P0nZpZDfn5Jap
Nr6C01nw7LGWOP7XSEAqQOZJrRZdZVqJhJhFpbYML7uT4BpVzAeMdCnScRLr89nSEzLzFTztZgxz
SIgGDbZ/X1x1m9hCzEyQNrJQh+wUjGzIoXYJVmN3Q5mqudUdiBn0/lIkOqWmsszoB7Ipargs7TN5
FqR4hXMp6f6t24nEmq+ImELc4e1SeJ5HLEZtcsxsHmCuWeIU86mDN+/uQeOzbzcBaxdpZ1kym6N1
TWyniB0jFqWzKDjsZUIDVa69xN6g0AP+vp7pX8y2YFtrbeiBbA0bguNDMHV6DnwtvkFD5/sNRVwm
bkLU73jgKetjYUruw9Chw5zFNTh0P8F2eX3ovV9vQbbaktB31htsurfRYOtj6mdPa80VfpxsxBXP
Jw/B1/ZbjCeDKzq2pbXOocD3vL5YwDz9C6R8xDtsM0cl4hC3x+JJ12k94ntrtsANLi7AAiqSuO6x
iKh1j4igDvUHvVzk13xZHhlXFxnVQuYyDWVY3Ms2Wl/lCGoLutaN+AHfjOTNqTMEyiAxaLkWbHGZ
Bq29zkfl96Wki95zec1JX2BOW4Myum4uwtMfsrTMYMTEAEpXsPeGD5m+bL/vdpkxHUUo7P0mu9RG
YLlnrGWFgTAGVIsog2gpn+EVKJItHMpQZ/WPTzLOktw1XMZouFhifbBItGauzTd/u8GH3L35tN+2
CPR0neywDtEBp0U36ROxbA4vVrVKJ0KHcNT0rUeDyQkiem3rV4++eFUeBbeO68Uq13mZ06eez8kB
RVWSLWg903BwqD5qBk2QyksEL8SuZhyFFjIDzB0PlkqEmPy1vUi1FCfIhj58Eq3n7WuLvhJkaTKR
bymcO5sYoABcgaHc6ZAMsc9Dmyl0YhGVkcTW0cmHTyan8E9+/fX9yau3hGI6+vDq49tXX04+/+90
tGzBzB/GrPg0fTi01Zs17ci2NOsEqWrQuZke3nZXZJukg3h1IYb0iouhS841LttikI5yDn6hs6Wz
Di15S4c+OF2h4PR0Nl7dne6WjsXMdjtbcs4eUCl0KgYUb5QLEeFnnZ3My7RPFp2/fC1hBPNnSdtG
HCwAbMIi1z5zr6mpq3xllj58xjw+PurMxP5WMp6G+5hD6S5rqymTjXux605HxjkyAcshPMyu7Zg0
bV1cNNLBZv9sqpGDrjpMg3p7ltI0uIf1Py/UGlrKObbnG5V5O4CRznyxFXIIZ33/qV2FMVltlT1v
rYdWvLuOgid0i7xpdyD9hB8UsxzfI1SYXLEp/fdbKNN3qUwXAfUoL6S3un4RxnJTcDvgmt3RpbvW
qceieSKhuAPkbc02Kj5wHIDZQIIR1yDIA+Nb1QQ4y9HJepTtaJdK44G2IqLdVQx8u3ity7Hd3T9q
Wd5yd5zjX0QN69JvSqN7WxOEQ8fA41BdjbN8gfhDkeb+ompVwWzdTToGClf6eRQemzMVGxM1oKnG
CQ5hnB5ostONZicrf1M4/EetwunUtO5Kb/6wF4nnlB/YHv0ddwKXa/3hjY365ibuyO1uhLvptA36
kpnCm7JDmIRy7mtbBx6wUNEpb2I6Je3VzHyCQS0TPY2Ef01HcmDEvBPKqS3vRXVaODyJ/wYjkll1
fNCNHx6fDMG76NPqbqXfIFDO3eUh7+vyrI2Vjs24xjbFYC9JX1xKLXgSgJadhd1+J+G6zCtGoPer
e3AA9X3C7Pt0ujgd+T615HwflmW6j9yhm2aj5X8A/C8QpWk+AAA=
""")).decode())
for name, text in FILES.items():
    open(os.path.join(ROOT, name), 'w').write(text)
print('working in', ROOT)


In [ ]:
%%bash
nvidia-smi --query-gpu=name,compute_cap,driver_version,memory.total --format=csv
nproc; grep -m1 'model name' /proc/cpuinfo; lscpu | grep -E '^(Thread|Core|Socket)'


In [ ]:
%%bash
# NVIDIA HPC SDK (nvc -mp=gpu): the compiler the GPU evaluation uses (NOTES.md, 2026-10-02).
set -e
if ls /opt/nvidia/hpc_sdk/Linux_x86_64/*/compilers/bin/nvc >/dev/null 2>&1; then echo "already installed"; exit 0; fi
curl -s https://developer.download.nvidia.com/hpc-sdk/ubuntu/DEB-GPG-KEY-NVIDIA-HPC-SDK \
  | gpg --dearmor --yes -o /usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg
echo 'deb [signed-by=/usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg] https://developer.download.nvidia.com/hpc-sdk/ubuntu/amd64 /' \
  > /etc/apt/sources.list.d/nvhpc.list
apt-get update -qq
PKG=$(apt-cache search --names-only '^nvhpc-[0-9]+-[0-9]+$' | awk '{print $1}' | sort -V | tail -1)
echo "Installing $PKG"; apt-get install -y -qq $PKG > /tmp/nvhpc.log 2>&1 || { tail -20 /tmp/nvhpc.log; exit 1; }
ls -d /opt/nvidia/hpc_sdk/Linux_x86_64/*/compilers/bin/nvc


In [ ]:
%%bash
cd "$P05_ROOT"
NAME="$P05_PLATFORM-t4"
NVC=$(ls -d /opt/nvidia/hpc_sdk/Linux_x86_64/*/compilers/bin/nvc | sort -V | tail -1)
CC=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d .)
python3 calibrate.py --name "$NAME" --gpu --cc "$NVC" --cflags "-O2 -mp=gpu -gpu=cc$CC" \
  --notes "$P05_PLATFORM $(nvidia-smi --query-gpu=name --format=csv,noheader | head -1), host $(nproc) vCPU; nvc -mp=gpu" \
  --out "$NAME.json"
echo; echo "===== copy everything below this line back to Claude ====="
cat "$NAME.json"
